# Fine-tune on CES Survey Data (QLoRA, Colab)

**Runtime → Change runtime type → GPU**

Edit the CONFIG cell below to change model, condition, and training params.

**To pull latest changes** after initial clone:
```python
!git -C article_silicon_sampling_quebec pull origin main
```

In [ ]:
# ============================================================
# CONFIG — edit these before running
# ============================================================

# Your HF token (https://huggingface.co/settings/tokens)
HF_TOKEN = ""  # <-- SET YOUR TOKEN HERE

# Model options (BASE models — not Instruct):
#   meta-llama/Llama-3.2-1B              (~4GB VRAM)
#   meta-llama/Llama-3.2-3B              (~8GB VRAM, RECOMMENDED)
#   Qwen/Qwen2.5-0.5B                    (~3GB VRAM)
#   Qwen/Qwen2.5-1.5B                    (~5GB VRAM)
#   Qwen/Qwen2.5-3B                      (~8GB VRAM)
MODEL_NAME = "meta-llama/Llama-3.2-3B"

# Dataset condition (from huggingface.co/datasets/hubcad25/article_silicon_sampling_quebec_data)
# Options: finetune_train_4a.jsonl, 4b.jsonl, 5a.jsonl, 5b.jsonl
DATASET_FILE = "finetune_train_4a.jsonl"

# Training params
EPOCHS = 1          # 1 for smoke test, 3 for full run
MAX_TRAIN_SAMPLES = 5000  # Remove/comment for full dataset
HF_REPO = "hubcad25/llama-3.2-3b-quebec-lora-condition4a"  # Output HF repo
# ============================================================
import os
os.environ["HF_TOKEN"] = HF_TOKEN
print(f"Model: {MODEL_NAME}")
print(f"Dataset: {DATASET_FILE}")
print(f"Epochs: {EPOCHS}, max_train_samples: {MAX_TRAIN_SAMPLES or 'ALL'}")

In [ ]:
# Install dependencies
!pip install -q unsloth "unsloth[cu124-torch241]" trl transformers accelerate peft datasets huggingface_hub
!pip install -q polars

In [ ]:
# Clone repo + download dataset
!git clone https://github.com/hubcad25/article_silicon_sampling_quebec.git
import os
os.chdir("article_silicon_sampling_quebec")

from huggingface_hub import hf_hub_download
dataset_local = hf_hub_download(
    repo_id="hubcad25/article_silicon_sampling_quebec_data",
    filename=DATASET_FILE,
    token=HF_TOKEN,
    repo_type="dataset",
)
print(f"Dataset: {dataset_local}")
!wc -l {dataset_local}

In [ ]:
# Run fine-tuning
OUTPUT_DIR = "/content/model_output"
import subprocess

cmd = [
    "python", "scripts/finetune.py",
    "--data", dataset_local,
    "--model", MODEL_NAME,
    "--output_dir", OUTPUT_DIR,
    "--use_4bit",
    "--epochs", str(EPOCHS),
    "--batch_size", "4",
    "--grad_accum", "8",
    "--lr", "2e-4",
    "--max_seq_len", "2048",
    "--hf_repo", HF_REPO,
]
if MAX_TRAIN_SAMPLES:
    cmd.extend(["--max_train_samples", str(MAX_TRAIN_SAMPLES)])

env = os.environ.copy()
result = subprocess.run(cmd, env=env)
print("Done." if result.returncode == 0 else f"Failed: {result.returncode}")